# 04 — Remaining useful life

Gradient boosting on rolling features, plus an LSTM on cycle windows.


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from src.data_preprocessing import SENSOR_COLUMNS, prepare_datasets
from src.feature_engineering import add_health_index, add_rolling_features, sequence_windows
from src.rul_prediction import train_gbr, train_lstm, save_gbr, save_lstm
from src.evaluation import regression_report, nasa_score

train, test, _ = prepare_datasets()
train_f = add_health_index(add_rolling_features(train))
test_f = add_health_index(add_rolling_features(test))
skip = {"unit_id", "cycle", "RUL", "failure", "fail_mode"}
feature_cols = [c for c in train_f.columns if c not in skip]

In [ ]:
X_tr = train_f[feature_cols].to_numpy(dtype=np.float32)
y_tr = train_f["RUL"].to_numpy(dtype=np.float32)
X_te = test_f[feature_cols].to_numpy(dtype=np.float32)
y_te = test_f["RUL"].to_numpy(dtype=np.float32)

gbr = train_gbr(X_tr, y_tr)
pred = gbr.predict(X_te)
print(regression_report(y_te, pred))
print("NASA score", nasa_score(y_te, pred))
save_gbr(gbr)

In [ ]:
WINDOW = 20
Xseq_tr, yseq_tr = sequence_windows(train_f, SENSOR_COLUMNS, window=WINDOW)
Xseq_te, yseq_te = sequence_windows(test_f, SENSOR_COLUMNS, window=WINDOW)
print(Xseq_tr.shape, Xseq_te.shape)

lstm = train_lstm(Xseq_tr, yseq_tr, epochs=15)
lstm_pred = lstm.predict(Xseq_te, verbose=0).ravel()
print(regression_report(yseq_te, lstm_pred))
print("NASA score", nasa_score(yseq_te, lstm_pred))
save_lstm(lstm)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(y_te, pred, s=8, alpha=0.3)
lims = [0, max(y_te.max(), pred.max())]
ax.plot(lims, lims, color="red")
ax.set_xlabel("True RUL")
ax.set_ylabel("Predicted RUL (GBR)")
ax.set_title("RUL scatter")
plt.show()